# 长音频四模型对比 (VAD)

使用 VAD 分段 + 四模型串行评估长录音。

In [1]:
import warnings, logging, os, re, json, time, gc, torch
logging.getLogger('root').setLevel(logging.ERROR)
warnings.filterwarnings('ignore')

device = 'mps' if torch.backends.mps.is_available() else 'cpu'
print(f'PyTorch: {torch.__version__}, Device: {device}')

DATA_ROOT = os.path.expanduser('~/Desktop/hengdong_asr_trainset')
SV_CKPT = os.path.expanduser('~/Projects/Agent/model/sensevoice_lora/model.pt.best')
NANO_CKPT = os.path.expanduser('~/Projects/Agent/model/funasr_nano_v3/model.pt.best')

def clean_sensevoice_text(text):
    return re.sub(r'<\|[^|]*\|>', '', text).strip()

def levenshtein(s1, s2):
    if len(s1) < len(s2): return levenshtein(s2, s1)
    if len(s2) == 0: return len(s1)
    prev = list(range(len(s2) + 1))
    for c1 in s1:
        curr = [prev[0] + 1]
        for j, c2 in enumerate(s2):
            curr.append(min(prev[j + 1] + 1, curr[j] + 1, prev[j] + (c1 != c2)))
        prev = curr
    return prev[-1]

_PUNCT = re.compile(r'[\s\.,!?;:\-()\[\]{}\u3000\uff0c\u3002\uff01\uff1f\u3001\uff1b\uff1a\u201c\u201d\u2018\u2019\uff08\uff09\u3010\u3011\u300a\u300b\u2026\u2014]')
def norm_punct(text): return _PUNCT.sub('', text)

def free_memory():
    gc.collect()
    if torch.backends.mps.is_available(): torch.mps.empty_cache()

print('工具函数就绪')

PyTorch: 2.11.0, Device: mps
工具函数就绪


In [2]:
# 加载长音频数据
LONG_JSONL = os.path.join(DATA_ROOT, 'manifests/extra_long_talks.jsonl')
long_samples = []
with open(LONG_JSONL) as f:
    for line in f:
        item = json.loads(line)
        audio = item['audio_filepath']
        if audio.startswith('/mnt/data/'):
            audio = audio.replace('/mnt/data/hengdong_asr_trainset', DATA_ROOT)
        long_samples.append({
            'id': item['id'],
            'audio': audio,
            'text': item['text'],
            'duration': item['duration'],
            'speaker': item.get('speaker_id', 'unknown'),
        })
print(f'长音频: {len(long_samples)} 条')
for s in long_samples:
    exists = os.path.exists(s['audio'])
    print(f'  {s["speaker"]} | {s["duration"]/60:.1f}min | {"OK" if exists else "NOT FOUND"}')

长音频: 6 条
  talks_方言老女 | 27.1min | OK
  talks_方言老男 | 30.8min | OK
  talks_方言青女 | 23.3min | OK
  talks_方言青男 | 23.3min | OK
  dialogs_方言多人 | 16.7min | OK
  dialogs_方言多人 | 9.8min | OK


In [3]:
VAD_MODEL = 'iic/speech_fsmn_vad_zh-cn-16k-common-pytorch'
VAD_KWARGS = {'max_single_segment_time': 30000}

def eval_long_sensevoice(model, long_samples, label):
    """评估长音频 SenseVoice 模型 (VAD 内置)"""
    from funasr import AutoModel
    results = []
    start = time.time()
    for i, s in enumerate(long_samples):
        if not os.path.exists(s['audio']): continue
        print(f'  {label}: [{i+1}/{len(long_samples)}] {s["speaker"]} ({s["duration"]/60:.1f}min)')
        try:
            res = model.generate(input=s['audio'], language='auto', use_itn=True)
            pred_text = ''.join(clean_sensevoice_text(r['text']) for r in res if 'text' in r)
        except Exception as e:
            print(f'    Error: {e}')
            pred_text = ''
        exp_n = norm_punct(s['text'])
        pred_n = norm_punct(pred_text)
        cer = levenshtein(exp_n, pred_n) / max(len(exp_n), 1)
        results.append({
            'id': s['id'], 'speaker': s['speaker'], 'audio': s['audio'],
            'duration_min': s['duration'] / 60, 'cer': cer,
            'ref_len': len(exp_n), 'pred_len': len(pred_n),
            'expected': s['text'], 'predicted': pred_text,
        })
        print(f'    CER: {cer:.1%} | ref: {results[-1]["ref_len"]} | pred: {results[-1]["pred_len"]}')
    elapsed = time.time() - start
    avg_cer = sum(r['cer'] for r in results) / len(results) if results else 0
    return {'name': label, 'results': results, 'avg_cer': avg_cer, 'total': len(results), 'time': elapsed}

def eval_long_nano(asr_model, long_samples, label, direct=False):
    """评估长音频 Nano 模型
    - direct=False: 手动 VAD 分段 + 逐段推理
    - direct=True: 不分段，直接整音频传入模型（依赖模型内置 VAD）
    注意: Nano 不支持内置 VAD，需要手动 VAD 分段后逐段识别
    使用 soundfile 读取音频以避免 Mac 上 torchaudio 的兼容性问题
    """
    import soundfile as sf
    results = []
    start = time.time()
    with torch.inference_mode():
        for i, s in enumerate(long_samples):
            if not os.path.exists(s['audio']): continue
            mode_str = '不分段直接推理' if direct else 'VAD分段推理'
            print(f'  {label}: [{i+1}/{len(long_samples)}] {s["speaker"]} ({s["duration"]/60:.1f}min) [{mode_str}]')
            try:
                if direct:
                    # 方式1: 直接整音频传入（不分段）
                    res = asr_model.generate(input=s['audio'], language='auto', itn=True)
                    if res and 'text' in res[0]:
                        pred_text = res[0]['text']
                    else:
                        pred_text = ''
                    print(f'    直接推理完成, 输出长度: {len(pred_text)}')
                else:
                    # 方式2: VAD 分段后逐段推理
                    vad_model = AutoModel(model=VAD_MODEL, disable_update=True, device=device)
                    vad_res = vad_model.generate(input=s['audio'])
                    del vad_model
                    free_memory()
                    
                    segments = vad_res[0]['value'] if vad_res and 'value' in vad_res[0] else []
                    if not segments:
                        print(f'    VAD 未检测到语音段')
                        continue
                    print(f'    VAD 检测到 {len(segments)} 个语音段')

                    waveform, sr = sf.read(s['audio'], dtype='float32')
                    if waveform.ndim > 1:
                        waveform = waveform.squeeze()

                    pred_parts = []
                    for seg_idx, seg in enumerate(segments):
                        start_ms = int(seg[0])
                        end_ms = int(seg[1])
                        duration_ms = end_ms - start_ms
                        if duration_ms < 300:
                            continue

                        start_sample = int(start_ms * sr / 1000)
                        end_sample = int(end_ms * sr / 1000)
                        chunk = waveform[start_sample:end_sample]

                        if len(chunk) < sr * 0.3:
                            continue

                        import tempfile
                        with tempfile.NamedTemporaryFile(suffix='.wav', delete=False) as tmp:
                            tmp_path = tmp.name
                        try:
                            sf.write(tmp_path, chunk, sr)
                            res = asr_model.generate(input=tmp_path, language='auto', itn=True)
                            if res and 'text' in res[0]:
                                pred_parts.append(res[0]['text'])
                        finally:
                            if os.path.exists(tmp_path):
                                os.unlink(tmp_path)

                        if (seg_idx + 1) % 100 == 0:
                            print(f'    [{seg_idx+1}/{len(segments)}] 已处理 {len(pred_parts)} 段')

                    pred_text = ''.join(pred_parts)
                    print(f'    识别完成: {len(pred_parts)}/{len(segments)} 段, 输出长度: {len(pred_text)}')

            except Exception as e:
                print(f'    Error: {e}')
                import traceback
                traceback.print_exc()
                pred_text = ''

            exp_n = norm_punct(s['text'])
            pred_n = norm_punct(pred_text)
            cer = levenshtein(exp_n, pred_n) / max(len(exp_n), 1)
            results.append({
                'id': s['id'], 'speaker': s['speaker'], 'audio': s['audio'],
                'duration_min': s['duration'] / 60, 'cer': cer,
                'ref_len': len(exp_n), 'pred_len': len(pred_n),
                'expected': s['text'], 'predicted': pred_text,
            })
            print(f'    CER: {cer:.1%} | ref: {results[-1]["ref_len"]} | pred: {results[-1]["pred_len"]}')

    elapsed = time.time() - start
    avg_cer = sum(r['cer'] for r in results) / len(results) if results else 0
    return {'name': label, 'results': results, 'avg_cer': avg_cer, 'total': len(results), 'time': elapsed}

print('长音频评估函数就绪')

长音频评估函数就绪


### 注意
长音频的参考文本是**摘要整理版**，而模型输出是**逐字转录版**，两者粒度不同。
因此 CER 偏高是正常现象，主要作为模型间横向对比参考。

### [1/4] SV-base (长音频)

In [4]:
from funasr import AutoModel

long_all_results = {}

print('[1/4] SV-base (长音频)')
long_model = AutoModel(
    model='iic/SenseVoiceSmall', disable_update=True, device=device,
    vad_model=VAD_MODEL, vad_kwargs=VAD_KWARGS,
)
long_all_results['SV-base'] = eval_long_sensevoice(long_model, long_samples, 'SV-base')
print(f'  平均CER={long_all_results["SV-base"]["avg_cer"]:.1%} | 耗时={long_all_results["SV-base"]["time"]:.1f}s')

del long_model
free_memory()

[1/4] SV-base (长音频)
funasr version: 1.3.1.
  SV-base: [1/6] talks_方言老女 (27.1min)


100%|██████████| 141/141 [00:02<00:00, 65.42it/s]
{'load_data': '0.000', 'extract_feat': '0.188', 'forward': '2.155', 'batch_size': '141', 'rtf': '0.010'}, : 100%|██████████| 141/141 [00:02<00:00, 65.42it/s]
rtf_avg: 0.010: 100%|██████████| 141/141 [00:02<00:00, 65.37it/s]                                                                                            

100%|██████████| 94/94 [00:01<00:00, 48.76it/s]
{'load_data': '0.000', 'extract_feat': '0.153', 'forward': '1.928', 'batch_size': '94', 'rtf': '0.008'}, : 100%|██████████| 94/94 [00:01<00:00, 48.76it/s]
rtf_avg: 0.008: 100%|██████████| 94/94 [00:01<00:00, 48.72it/s]                                                                                           

100%|██████████| 67/67 [00:01<00:00, 36.27it/s]
{'load_data': '0.000', 'extract_feat': '0.142', 'forward': '1.847', 'batch_size': '67', 'rtf': '0.007'}, : 100%|██████████| 67/67 [00:01<00:00, 36.27it/s]
rtf_avg: 0.007: 100%|██████████| 67/67 [00:01<00:00, 36.24it/s]        

    CER: 419.9% | ref: 909 | pred: 4108
  SV-base: [2/6] talks_方言老男 (30.8min)


100%|██████████| 179/179 [00:02<00:00, 86.93it/s]
{'load_data': '0.000', 'extract_feat': '0.188', 'forward': '2.059', 'batch_size': '179', 'rtf': '0.009'}, : 100%|██████████| 179/179 [00:02<00:00, 86.93it/s]
rtf_avg: 0.009: 100%|██████████| 179/179 [00:02<00:00, 86.85it/s]                                                                                            

100%|██████████| 128/128 [00:01<00:00, 66.90it/s]
{'load_data': '0.000', 'extract_feat': '0.175', 'forward': '1.913', 'batch_size': '128', 'rtf': '0.008'}, : 100%|██████████| 128/128 [00:01<00:00, 66.90it/s]
rtf_avg: 0.008: 100%|██████████| 128/128 [00:01<00:00, 66.81it/s]                                                                                            

100%|██████████| 90/90 [00:01<00:00, 50.25it/s]
{'load_data': '0.000', 'extract_feat': '0.169', 'forward': '1.791', 'batch_size': '90', 'rtf': '0.007'}, : 100%|██████████| 90/90 [00:01<00:00, 50.25it/s]
rtf_avg: 0.007: 100%|██████████| 90/90 [00:01<00:00, 50.18it/s]

    CER: 253.9% | ref: 1268 | pred: 3745
  SV-base: [3/6] talks_方言青女 (23.3min)


...

    CER: 285.4% | ref: 1515 | pred: 4638
  SV-base: [5/6] dialogs_方言多人 (16.7min)


100%|██████████| 28/28 [00:02<00:00, 11.42it/s]
{'load_data': '0.000', 'extract_feat': '0.082', 'forward': '2.452', 'batch_size': '28', 'rtf': '0.018'}, : 100%|██████████| 28/28 [00:02<00:00, 11.42it/s]
rtf_avg: 0.018: 100%|██████████| 28/28 [00:02<00:00, 11.41it/s]                                                                                           

100%|██████████| 14/14 [00:03<00:00,  4.33it/s]
{'load_data': '0.000', 'extract_feat': '0.141', 'forward': '3.234', 'batch_size': '14', 'rtf': '0.012'}, : 100%|██████████| 14/14 [00:03<00:00,  4.33it/s]
rtf_avg: 0.012: 100%|██████████| 14/14 [00:03<00:00,  4.33it/s]                                                                                           

100%|██████████| 10/10 [00:02<00:00,  4.76it/s]
{'load_data': '0.000', 'extract_feat': '0.160', 'forward': '2.102', 'batch_size': '10', 'rtf': '0.007'}, : 100%|██████████| 10/10 [00:02<00:00,  4.76it/s]
rtf_avg: 0.007: 100%|██████████| 10/10 [00:02<00:00,  4.75it/s]                

    CER: 450.5% | ref: 311 | pred: 1466
  SV-base: [6/6] dialogs_方言多人 (9.8min)


100%|██████████| 22/22 [00:02<00:00, 10.72it/s]
{'load_data': '0.000', 'extract_feat': '0.094', 'forward': '2.052', 'batch_size': '22', 'rtf': '0.013'}, : 100%|██████████| 22/22 [00:02<00:00, 10.72it/s]
rtf_avg: 0.013: 100%|██████████| 22/22 [00:02<00:00, 10.71it/s]                                                                                           

100%|██████████| 10/10 [00:05<00:00,  1.76it/s]
{'load_data': '0.000', 'extract_feat': '0.128', 'forward': '5.691', 'batch_size': '10', 'rtf': '0.022'}, : 100%|██████████| 10/10 [00:05<00:00,  1.76it/s]
rtf_avg: 0.022: 100%|██████████| 10/10 [00:05<00:00,  1.76it/s]                                                                                           

100%|██████████| 5/5 [00:01<00:00,  4.10it/s]
{'load_data': '0.000', 'extract_feat': '0.094', 'forward': '1.220', 'batch_size': '5', 'rtf': '0.008'}, : 100%|██████████| 5/5 [00:01<00:00,  4.10it/s]
rtf_avg: 0.008: 100%|██████████| 5/5 [00:01<00:00,  4.09it/s]                       

    CER: 576.5% | ref: 204 | pred: 1256
  平均CER=456.7% | 耗时=69.6s


### [2/4] SV-ft (微调) (长音频)

In [5]:
print('[2/4] SV-ft (长音频)')
long_model = AutoModel(
    model='iic/SenseVoiceSmall', lora_only=True, disable_update=True, device=device,
    vad_model=VAD_MODEL, vad_kwargs=VAD_KWARGS,
)
long_ckpt = torch.load(SV_CKPT, map_location='cpu', weights_only=False)
long_model.model.load_state_dict(long_ckpt['state_dict'], strict=False)
del long_ckpt
print('  微调权重已载入')

long_all_results['SV-ft'] = eval_long_sensevoice(long_model, long_samples, 'SV-ft')
print(f'  平均CER={long_all_results["SV-ft"]["avg_cer"]:.1%} | 耗时={long_all_results["SV-ft"]["time"]:.1f}s')

del long_model
free_memory()

[2/4] SV-ft (长音频)
funasr version: 1.3.1.
  微调权重已载入
  SV-ft: [1/6] talks_方言老女 (27.1min)


100%|██████████| 141/141 [00:01<00:00, 72.50it/s]
{'load_data': '0.000', 'extract_feat': '0.173', 'forward': '1.945', 'batch_size': '141', 'rtf': '0.009'}, : 100%|██████████| 141/141 [00:01<00:00, 72.50it/s]
rtf_avg: 0.009: 100%|██████████| 141/141 [00:01<00:00, 72.44it/s]                                                                                            

100%|██████████| 94/94 [00:01<00:00, 51.26it/s]
{'load_data': '0.000', 'extract_feat': '0.162', 'forward': '1.834', 'batch_size': '94', 'rtf': '0.007'}, : 100%|██████████| 94/94 [00:01<00:00, 51.26it/s]
rtf_avg: 0.007: 100%|██████████| 94/94 [00:01<00:00, 51.22it/s]                                                                                           

100%|██████████| 67/67 [00:01<00:00, 38.23it/s]
{'load_data': '0.000', 'extract_feat': '0.165', 'forward': '1.752', 'batch_size': '67', 'rtf': '0.007'}, : 100%|██████████| 67/67 [00:01<00:00, 38.23it/s]
rtf_avg: 0.007: 100%|██████████| 67/67 [00:01<00:00, 38.20it/s]        

    CER: 392.8% | ref: 909 | pred: 3983
  SV-ft: [2/6] talks_方言老男 (30.8min)


100%|██████████| 179/179 [00:02<00:00, 84.75it/s]
{'load_data': '0.000', 'extract_feat': '0.226', 'forward': '2.112', 'batch_size': '179', 'rtf': '0.009'}, : 100%|██████████| 179/179 [00:02<00:00, 84.75it/s]
rtf_avg: 0.009: 100%|██████████| 179/179 [00:02<00:00, 84.67it/s]                                                                                            

100%|██████████| 128/128 [00:01<00:00, 67.97it/s]
{'load_data': '0.000', 'extract_feat': '0.217', 'forward': '1.883', 'batch_size': '128', 'rtf': '0.007'}, : 100%|██████████| 128/128 [00:01<00:00, 67.97it/s]
rtf_avg: 0.007: 100%|██████████| 128/128 [00:01<00:00, 67.91it/s]                                                                                            

100%|██████████| 90/90 [00:01<00:00, 48.70it/s]
{'load_data': '0.000', 'extract_feat': '0.203', 'forward': '1.848', 'batch_size': '90', 'rtf': '0.007'}, : 100%|██████████| 90/90 [00:01<00:00, 48.70it/s]
rtf_avg: 0.007: 100%|██████████| 90/90 [00:01<00:00, 48.64it/s]

    CER: 219.6% | ref: 1268 | pred: 3624
  SV-ft: [3/6] talks_方言青女 (23.3min)


...

    CER: 272.7% | ref: 1515 | pred: 4497
  SV-ft: [5/6] dialogs_方言多人 (16.7min)


100%|██████████| 28/28 [00:02<00:00, 12.46it/s]
{'load_data': '0.000', 'extract_feat': '0.089', 'forward': '2.248', 'batch_size': '28', 'rtf': '0.016'}, : 100%|██████████| 28/28 [00:02<00:00, 12.46it/s]
rtf_avg: 0.016: 100%|██████████| 28/28 [00:02<00:00, 12.44it/s]                                                                                           

100%|██████████| 14/14 [00:03<00:00,  4.18it/s]
{'load_data': '0.000', 'extract_feat': '0.155', 'forward': '3.347', 'batch_size': '14', 'rtf': '0.013'}, : 100%|██████████| 14/14 [00:03<00:00,  4.18it/s]
rtf_avg: 0.013: 100%|██████████| 14/14 [00:03<00:00,  4.18it/s]                                                                                           

100%|██████████| 10/10 [00:02<00:00,  4.91it/s]
{'load_data': '0.000', 'extract_feat': '0.154', 'forward': '2.038', 'batch_size': '10', 'rtf': '0.007'}, : 100%|██████████| 10/10 [00:02<00:00,  4.91it/s]
rtf_avg: 0.007: 100%|██████████| 10/10 [00:02<00:00,  4.90it/s]                

    CER: 435.7% | ref: 311 | pred: 1445
  SV-ft: [6/6] dialogs_方言多人 (9.8min)


100%|██████████| 22/22 [00:01<00:00, 11.58it/s]
{'load_data': '0.000', 'extract_feat': '0.088', 'forward': '1.900', 'batch_size': '22', 'rtf': '0.012'}, : 100%|██████████| 22/22 [00:01<00:00, 11.58it/s]
rtf_avg: 0.012: 100%|██████████| 22/22 [00:01<00:00, 11.57it/s]                                                                                           

100%|██████████| 10/10 [00:02<00:00,  3.35it/s]
{'load_data': '0.000', 'extract_feat': '0.127', 'forward': '2.981', 'batch_size': '10', 'rtf': '0.012'}, : 100%|██████████| 10/10 [00:02<00:00,  3.35it/s]
rtf_avg: 0.012: 100%|██████████| 10/10 [00:02<00:00,  3.35it/s]                                                                                           

100%|██████████| 5/5 [00:01<00:00,  4.19it/s]
{'load_data': '0.000', 'extract_feat': '0.085', 'forward': '1.195', 'batch_size': '5', 'rtf': '0.008'}, : 100%|██████████| 5/5 [00:01<00:00,  4.19it/s]
rtf_avg: 0.008: 100%|██████████| 5/5 [00:01<00:00,  4.18it/s]                       

    CER: 683.3% | ref: 204 | pred: 1504
  平均CER=477.2% | 耗时=64.8s


### [3/4] Nano-base (长音频)

In [6]:
# print('[3/4] Nano-base (长音频, VAD分段)')
# nano_base = AutoModel(model='FunAudioLLM/Fun-ASR-Nano-2512', hub='ms', disable_update=True, device=device)

# long_all_results['Nano-base'] = eval_long_nano(nano_base, long_samples, 'Nano-base', direct=False)
# print(f'  平均CER={long_all_results["Nano-base"]["avg_cer"]:.1%} | 耗时={long_all_results["Nano-base"]["time"]:.1f}s')

# del nano_base
# free_memory()

[3/4] Nano-base (长音频, VAD分段)
funasr version: 1.3.1.
  Nano-base: [1/6] talks_方言老女 (27.1min) [VAD分段推理]
funasr version: 1.3.1.


rtf_avg: 0.190: 100%|██████████| 1/1 [00:01<00:00,  1.59s/it]                                                                                          


    VAD 检测到 373 个语音段


rtf_avg: 0.228: 100%|██████████| 1/1 [00:00<00:00,  2.15it/s]                                                                                          


    [100/373] 已处理 100 段


...

    VAD 检测到 477 个语音段


rtf_avg: 0.205: 100%|██████████| 1/1 [00:00<00:00,  1.84it/s]                                                                                          


    [100/477] 已处理 100 段


  0%|          | 0/1 [00:00<?, ?it/s]

KeyboardInterrupt: 

### [4/4] Nano-ft (微调) (长音频)

In [ ]:
# print('[4/4] Nano-ft (长音频, VAD分段)')
# nano_ft = AutoModel(
#     model='FunAudioLLM/Fun-ASR-Nano-2512', hub='ms', disable_update=True, device=device,
#     init_param=NANO_CKPT,
# )
# print('  微调权重已载入')

# long_all_results['Nano-ft'] = eval_long_nano(nano_ft, long_samples, 'Nano-ft', direct=False)
# print(f'  平均CER={long_all_results["Nano-ft"]["avg_cer"]:.1%} | 耗时={long_all_results["Nano-ft"]["time"]:.1f}s')

# del nano_ft
# free_memory()